In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


In [ ]:
# Load the walmart dataset

walmart_dataset = pd.read_csv('Walmart_customer_purchases.csv')

In [ ]:
# Display first 5 rows

walmart_dataset.head()


In [ ]:
# Display last 5 rows

walmart_dataset.tail()

In [ ]:
# Check Shape of the dataset

np.shape(walmart_dataset)

In [ ]:
# List all cloumn names

walmart_dataset.info()

In [ ]:
# Check missing data

walmart_dataset.isnull().sum()

In [ ]:
walmart_dataset.head()

In [ ]:
# Group the data by customerid

walmart_dataset.groupby('Customer_ID')

In [ ]:
# Calculate the metrics(Sum ->'total spend', count->'frequency', mean->'Average', count * mean -> 'spending score') with each customerid

customer_metrics = walmart_dataset.groupby('Customer_ID').agg({'Purchase_Amount': ['sum', 'count', 'mean'], 'Age': 'first',
    'Gender': 'first', 'City':'first', 'Category': 'first', 'Payment_Method': 'first','Discount_Applied': 'first', 'Rating': 'mean',  'Repeat_Customer': 'first'})
customer_metrics.columns = ['Total Spend', 'Frequency', 'Average', 'Age', 'Gender', 'City', 'Category', 'Payment_Method', 'Discount_Applied', 'Average Rating', 'Repeat Customer']



In [ ]:
# Total categories

customer_metrics['Category'].unique()

In [ ]:
# Encode Category into numbers because K-Means needs numbers
# Each product category gets a unique number
customer_metrics['Category Number'] = pd.Categorical(
    customer_metrics['Category']
).codes

customer_metrics['Category Number']

In [ ]:
# Encode gender to number

customer_metrics["Gender Number"] = customer_metrics["Gender"].map({"Male": 1, "Female": 0, 'Other': 2 })

In [ ]:
# Encode Repeat Customer

customer_metrics['Repeat Customer Number'] = customer_metrics['Repeat Customer'].map({'Yes': 1, 'No': 0})

In [ ]:
clustering_features = ['Total Spend', 'Frequency', 'Average', 'Age', 'Average Rating', 'Category Number', 'Gender Number']

In [ ]:
# Select only the numeric columns that K-Means will use for clustering
# These are the features that describe each customer's behavior
clustering_features = ['Total Spend', 'Frequency','Age']

# StandardScaler transforms all features to the same scale
# This ensures no single feature dominates the distance calculation
# For example: Total Spend can be 10,000 but Age is only 30
# After scaling, both will be on the same level
scaler = StandardScaler()

# fit transform learns the scale from data and transforms it in one step
# scaled customer data is a numpy array of normalized values
scaled_customer_data = scaler.fit_transform(customer_metrics[clustering_features])

In [ ]:
# Elbow Method: To find how many cluster I need to take instaed of just guessing it.

inertia = []
k_range = range(1,11)

for i in k_range:
    km = KMeans(n_clusters=i, random_state=42)
    km.fit(scaled_customer_data)
    inertia.append(km.inertia_)

plt.plot(k_range,inertia)
plt.xlabel("Number of cluster")
plt.ylabel("Inertia")
plt.title("Elbow Method: Finding Best K")
plt.show()


In [ ]:
# The above graph says k = 2 ans k = 3 has huge drop
# k = 4 has medium drop so taking k = 4 as my final cluster.

#Training final model

kmeans = KMeans(n_clusters=4, random_state=42)
kmeans.fit(scaled_customer_data)


In [ ]:
# Step 2 - Add cluster labels back to dataframe
customer_metrics['Cluster'] = kmeans.labels_

In [ ]:
# Visual Scatter Plot

sns.scatterplot(data=customer_metrics, x='Total Spend', y='Age', hue='Cluster', palette='viridis')
plt.title("Customer Segmentatiom")
plt.show()

In [ ]:

cluster_summary = customer_metrics.groupby('Cluster')[['Total Spend', 'Age', 'Average Rating']].mean()
print(cluster_summary)

In [ ]:
# Check actual numbers per cluster
verification = customer_metrics.groupby('Cluster')[
    ['Total Spend', 'Frequency', 'Average', 'Age', 'Average Rating']
].mean().round(2)

print(verification)

In [ ]:
cluster_names = {
    0: 'High Value Seniors',   # High spend (375) + Old age (49)
    1: 'Budget Seniors',       # Low spend (129)  + Old age (49)
    2: 'High Value Youth',     # High spend (381) + Young age (28)
    3: 'Budget Youth'          # Low spend (136)  + Young age (28)
}

customer_metrics['Cluster Name'] = customer_metrics['Cluster'].map(cluster_names)

print(customer_metrics['Cluster Name'].value_counts())

In [ ]:
# Understanding each cluster using all available data


for cluster in customer_metrics['Cluster Name'].unique():

    one_cluster  = customer_metrics[customer_metrics['Cluster Name'] == cluster]

    top_category = one_cluster['Category'].value_counts().index[0]
    top_payment  = one_cluster['Payment_Method'].value_counts().index[0]
    loyalty      = round(one_cluster['Repeat Customer Number'].mean() * 100, 1)

    print(f"\nCluster      : {cluster}")
    print(f"Top Category : {top_category}")
    print(f"Top Payment  : {top_payment}")
    print(f"Loyalty Rate : {loyalty}%")
    print("-" * 40)

In [ ]:
# Visual  Box Plot
sns.boxplot(data=customer_metrics, x='Cluster Name', y='Total Spend', palette='viridis')
plt.title("Spending Distribution per Cluster")
plt.xlabel("Cluster")
plt.ylabel("Total Spend")
plt.show()

In [ ]:
# Visual Count Plot
sns.countplot(data=customer_metrics, x='Cluster Name', hue='Cluster', palette='viridis')
plt.title("Number of Customers per Cluster")
plt.show()

In [ ]:
# Visual Bar Plot

cluster_summary.plot(kind='bar', color=['blue','green','orange','red'])
plt.title("Average Total Spend per Cluster")
plt.xlabel("Cluster")
plt.ylabel("Average Total Spend")
plt.show()

In [ ]:
# visual heatmap

verification = customer_metrics.groupby('Cluster Name')[['Total Spend', 'Age', 'Average Rating']].mean().round(2)
sns.heatmap(verification, annot=True, fmt='.1f', cmap='YlOrRd')
plt.title("Cluster Characteristics Heatmap")
plt.show()

In [ ]:
# Business Recommendations


print("\nBusiness Recommendations")
print("=" * 40)

for cluster in customer_metrics['Cluster Name'].unique():

    one_cluster = customer_metrics[customer_metrics['Cluster Name'] == cluster]

    top_category     = one_cluster['Category'].value_counts().index[0]
    top_payment     = one_cluster['Payment_Method'].value_counts().index[0]
    loyalty_pct = round(one_cluster['Repeat Customer Number'].mean() * 100, 1)
    avg_spend   = round(one_cluster['Total Spend'].mean(), 2)
    avg_age     = round(one_cluster['Age'].mean(), 2)

    print(f"\nCluster  : {cluster}")
    print(f"They buy : {top_category}")
    print(f"They pay : {top_payment}")
    print(f"Loyalty  : {loyalty_pct}%")

    if avg_spend > 300 and avg_age > 40:
        print(f"Action   : Give loyalty rewards + premium {top_category} offers")

    elif avg_spend < 200 and avg_age > 40:
        print(f"Action   : Send discount coupons on {top_category}")

    elif avg_spend > 300 and avg_age < 35:

        print(f"Action   : Promote trending {top_category} + cashback on {top_payment}")

    elif avg_spend < 200 and avg_age < 35:
        print(f"Action   : Student discounts on {top_category} + referral rewards")
    else:
        print(f"Action   : Offer personalised deals on {top_category}")

    print("-" * 40)